In [114]:
# IMPORTSSSS
import os
import glob
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, GlobalAveragePooling1D, Dense, Dropout, Concatenate, RepeatVector, TimeDistributed, Bidirectional, LSTM


In [115]:
# Hyperparameters
MAX_TIMESTEPS = 500  #  standardize all writing samples to 500 time-steps

# Delta_X (Stroke width)
# Delta_Y (Stroke height)
# Pressure (Stress/Force)
# Tilt_X (Pen grip)
# Tilt_Y (Pen grip)
# Velocity (Speed)
# Acceleration (Momentum)
# Jerk (Smoothness/Micro-stutters)
FEATURES = 8         

In [116]:

# Preprocess blocks
def analyze_stroke_data(csv_filepath):
    # 1. Load the data
    df = pd.read_csv(csv_filepath)
    
    # 2. Calculate time differences between rows (Delta Time / dt)
    df['dt'] = df['time'].diff().fillna(0)
    
    # 3. Calculate distance between points (Delta Distance using Pythagorean theorem)
    df['dx'] = df['x'].diff().fillna(0)
    df['dy'] = df['y'].diff().fillna(0)
    
    # 4. Calculate Velocity (Distance / Time)
    # np.where prevents division-by-zero errors if two events fire at the exact same millisecond
    df['distance'] = np.sqrt(df['dx']**2 + df['dy']**2) 
    df['velocity'] = np.where(df['dt'] > 0, df['distance'] / df['dt'], 0)
    
    # Calculate "Writing Duration" 
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 1
    writing_duration = df['dt'].where(df['touching'] == 1, 0).sum()

    # Calculate "In-Air Pen Duration" (The pause time biomarker)
    # Sum of the 'dt' column, but ONLY for rows where 'touching' == 0
    in_air_duration = df['dt'].where(df['touching'] == 0, 0).sum()
    
    # TODO 3: Print the results!
    print(f"--- Analysis for: {csv_filepath} ---")
    print(f"Total Writing Duration : {writing_duration} ms")
    print(f"Total In-Air Pauses    : {in_air_duration} ms")
    print(f"Average Pen Velocity   : {df['velocity'].mean():.2f} px/ms")
    print("-" * 40)
    
    return df


In [117]:
# PREPROCESSING and FEATURE EXTRACTION
def extract_golden_features(df):
    """Converts the raw CSV dataframe into the Golden 8 Features array."""
    # Safety checks for old files
    if "touching" not in df.columns: df["touching"] = True
    for col in ["tiltX", "tiltY", "latency"]:
        if col not in df.columns: df[col] = 0
        
    # Calculate the physics features
    df["dt"] = df["time"].diff().fillna(1)
    df.loc[df["dt"] == 0, "dt"] = 1
    
    df["delta_x"] = df["x"].diff().fillna(0)
    df["delta_y"] = df["y"].diff().fillna(0)
    df["distance"] = np.sqrt(df["delta_x"]**2 + df["delta_y"]**2)
    df["velocity"] = df["distance"] / df["dt"]
    df["acceleration"] = df["velocity"].diff().fillna(0) / df["dt"]
    df["jerk"] = df["acceleration"].diff().fillna(0) / df["dt"]
    
    # Extract EXACTLY our Golden 8 array
    golden_df = df[["delta_x", "delta_y", "pressure", "tiltX", "tiltY", "velocity", "acceleration", "jerk"]]
    
    # Fill any weird math errors (like dividing by zero) with 0
    golden_df = golden_df.fillna(0)
    return golden_df.values

In [118]:


    
    
def load_and_pad_data(data_dir="datasets/"):
    sequences = []
    latencies = []
    labels = []
    
    csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
    
    for file in csv_files:
        df = pd.read_csv(file)
        if len(df) == 0: continue
            
        # 1. Get Latency (from the very first row)
        latency_val = df["latency"].iloc[0] if "latency" in df.columns else 0
        
        # 2. Determine Label by reading the filename
        filename = os.path.basename(file).lower()
        if "normal" in filename:
            label = 0
        elif "dyslexia" in filename:
            label = 1
        else:
            continue # skip files that aren't labeled normal/dyslexia
            
        # 3. Use our new extractor! (Turns raw data into N x 8 array)
        stroke_data = extract_golden_features(df)
        
        # 4. Pad or Truncate to MAX_TIMESTEPS (500)
        if len(stroke_data) > MAX_TIMESTEPS:
            stroke_data = stroke_data[:MAX_TIMESTEPS] # Chop off excess
        else:
            padding = np.zeros((MAX_TIMESTEPS - len(stroke_data), FEATURES))
            stroke_data = np.vstack((stroke_data, padding)) # Add zeros to the end
            
        sequences.append(stroke_data)
        latencies.append(latency_val)
        # Expand the label to (500, 1)         
        labels.append(np.full((MAX_TIMESTEPS, 1), label))
        
    return np.array(sequences), np.array(latencies), np.array(labels)


In [119]:


def build_model():
    # kinematic sequence input
    sequence_input = Input(shape=(MAX_TIMESTEPS, FEATURES), name="kinematic_input")
    # Bidirectional reads the sequence forwards and backwards to understand context
    x = Bidirectional(LSTM(64, return_sequences=True))(sequence_input)
    x = Dropout(0.3)(x)
    x = Bidirectional(LSTM(32, return_sequences=True))(x)
    x = Dropout(0.3)(x)
    
    # latency input
    latency_input = Input(shape=(1,), name="latency_input")
    # Stretch the single Latency number across all 500 timesteps so it matches Branch A
    lat_repeated = RepeatVector(MAX_TIMESTEPS)(latency_input)
    
    # merge the inputs
    merged = Concatenate()([x, lat_repeated])
    output = TimeDistributed(Dense(1, activation='sigmoid'), name="heatmap_output")(merged)
    # Build and Compile
    
    model = Model(inputs=[sequence_input, latency_input], outputs=output, name='eldislexiav3',)
    model.compile(
        
        optimizer='adam', 
        loss='binary_crossentropy', 
        metrics=['accuracy', tf.keras.metrics.Recall(name='recall')]
    )
    return model

In [120]:
model = build_model()
model.summary()


Model: "eldislexiav3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ kinematic_input     │ (None, 500, 8)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_17    │ (None, 500, 128)  │     37,376 │ kinematic_input[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 500, 128)  │          0 │ bidirectional_17… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_18    │ (None, 500, 64)   │     41,216 │ dropout_18[0][0]  │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latency_input       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_19          │ (None, 500, 64)   │          0 │ bidirectional_18… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ repeat_vector_10    │ (None, 500, 1)    │          0 │ latency_input[0]… │
│ (RepeatVector)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_10      │ (None, 500, 65)   │          0 │ dropout_19[0][0], │
│ (Concatenate)       │                   │            │ repeat_vector_10… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ heatmap_output      │ (None, 500, 1)    │         66 │ concatenate_10[0… │
│ (TimeDistributed)   │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 78,658 (307.26 KB)

 Trainable params: 78,658 (307.26 KB)

 Non-trainable params: 0 (0.00 B)

In [121]:
print("Loading data...")
# Point this to your data collector folder
X_seq, X_lat, y = load_and_pad_data("datasets/") 
print(f"Data loaded! \nShape of X_seq (timeseries sequence) : {X_seq.shape} \nShape of X_lat (latency) :{X_lat.shape} \nShape of y (labels) : {y.shape}")
    

Loading data...
Data loaded! 
Shape of X_seq (timeseries sequence) : (14, 500, 8) 
Shape of X_lat (latency) :(14,) 
Shape of y (labels) : (14, 500, 1)


In [122]:

# Count how many total 0s and 1s exist in the entire training set 'y'
total_zeros = np.sum(y == 0)
total_ones = np.sum(y == 1)
total_samples = total_zeros + total_ones

# Apply the 'Weight = Total_Samples / (Number_of_Classes * Samples_in_Class)' formula
weight_for_0 = total_samples / (2.0 * total_zeros)
weight_for_1 = total_samples / (2.0 * total_ones)

# 3. Create the exact dictionary
sample_weight = np.ones(shape=y.shape) * weight_for_0
sample_weight[y == 1] = weight_for_1

print(f"Calculated Weights -> 0 (normal): {weight_for_0:.2f}, 1 (dyslexic): {weight_for_1:.2f}")

Calculated Weights -> 0 (normal): 1.00, 1 (dyslexic): 1.00


In [123]:

print("Starting training...")
history = model.fit([X_seq, X_lat], y, epochs=20, validation_split=0.2, sample_weight=sample_weight)
print("Saving the model...")
model.save("models/elkinematicV3.keras")

Starting training...
Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 23s 23s/step - accuracy: 0.6364 - loss: 5.8611 - recall: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 16.1181 - val_recall: 0.0000e+00
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 516ms/step - accuracy: 0.6364 - loss: 5.8611 - recall: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 16.1181 - val_recall: 0.0000e+00
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 307ms/step - accuracy: 0.6364 - loss: 5.8611 - recall: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 16.1181 - val_recall: 0.0000e+00
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 317ms/step - accuracy: 0.6364 - loss: 5.8611 - recall: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 16.1181 - val_recall: 0.0000e+00
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 286ms/step - accuracy: 0.6364 - loss: 5.8611 - recall: 0.0000e+00 - val_accuracy: 0.0000e+00 - val_loss: 16.1181 - val_recall: 0.0000e+00
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step - accuracy: 0.6364 - loss: 5.8611 - recall: 0